## pip3 install sentence-transformers

In [1]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

# MiniLM é um modelo leve e eficiente para criar embeddings de sentenças.
model = SentenceTransformer('all-MiniLM-L6-v2')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2685.28it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
frases = ["Eu gosto de SQL", "Eu odeio bugs"]
embeddings = model.encode(frases)


print(f"Quantidade de frases: {embeddings.shape[0]}")
print(f"Tamanho do vetor (Dimensões): {embeddings.shape[1]}") 
print("\nOs primeiros 10 números do vetor da primeira frase:")
print(embeddings[0][:10])
print(len(embeddings[0]))

Quantidade de frases: 2
Tamanho do vetor (Dimensões): 384

Os primeiros 10 números do vetor da primeira frase:
[ 0.04812893  0.0149713  -0.04699994 -0.03141594 -0.14765522  0.01044574
  0.09391669  0.06339914 -0.00590012 -0.03380473]
384


In [2]:
## Exemplo de Busca por Similaridade em FAQs

lista_faq = [
    "Como solicitar reembolso de compras",
    "Passo a passo para redefinir a senha",
    "Solução de problemas de congelamento de tela e lentidão",
    "Horário de atendimento do suporte",
    "O ponto agencia é lucrativo?"
]

# A Pergunta do Usuário (Query)
pergunta = input("Diga o seu problema: ") # "O app travou, o que faço?"


embeddings_faq = model.encode(lista_faq)
embedding_pergunta = model.encode(pergunta)

# Computes the cosine similarity between two tensors.
scores = util.cos_sim(embedding_pergunta, embeddings_faq)
print(f"Pergunta: {pergunta}\n")

# Criamos uma lista de pares (Score, Frase) e ordenamos
resultados = []
for i in range(len(lista_faq)):
    score_atual = scores[0][i].item() # .item() pega o número float
    resultados.append((score_atual, lista_faq[i]))

# Ordenar do maior para o menor
resultados.sort(key=lambda x: x[0], reverse=True)

for score, texto in resultados:
    print(f"Score: {score:.4f} | FAQ: {texto}")

Pergunta: eu vou ter lucro nesse local?

Score: 0.5913 | FAQ: O ponto agencia é lucrativo?
Score: 0.4181 | FAQ: Horário de atendimento do suporte
Score: 0.4034 | FAQ: Solução de problemas de congelamento de tela e lentidão
Score: 0.3922 | FAQ: Como solicitar reembolso de compras
Score: 0.2843 | FAQ: Passo a passo para redefinir a senha


In [3]:
produtos = ['iPhone 14 Pro', 'Samsung S23', 'Iphone 14 pro max 256gb', 'Galaxy S23 128gb', 'Fone Sony', 'Galaxy S23 plus']
emb_prod = model.encode(produtos)

matriz = util.cos_sim(emb_prod, emb_prod)

print("Produtos Duplicados detectados:")
for i in range(len(produtos)):
    for j in range(i + 1, len(produtos)):
        if matriz[i][j] > 0.75: 
            print(f"{produtos[i]}  <== É IGUAL ==>  {produtos[j]} (Score: {matriz[i][j]:.2f})")

Produtos Duplicados detectados:
Samsung S23  <== É IGUAL ==>  Galaxy S23 128gb (Score: 0.77)
Samsung S23  <== É IGUAL ==>  Galaxy S23 plus (Score: 0.88)
Galaxy S23 128gb  <== É IGUAL ==>  Galaxy S23 plus (Score: 0.79)


vamos pegar esse conceito e escalar. E se tivermos 1 milhão de PDFs? Não dá para calcular na hora igual fizemos aqui. Precisamos de um banco de dados especial para guardar esses números: O Vector Database (ChromaDB) e ligar isso a um LLM (Ollama)